# StarLayer User Guide (Notebook)

This notebook is located at https://github.com/hidden-graph/starlayer/blob/main/docs/user-guide-v1.ipynb

## How to run this notebook

1. pip install from the github repository.
2. Run cells from top to bottom so shared variables remain available.


In [ ]:
pip install "git+https://github.com/hidden-graph/starlayer.git"

In [16]:
from starlayergraph import StarLayerGraph, Namespace, TripleTerm, DirLangString, Literal
from starshacl import StarShaclValidator

EX = Namespace("http://example.org/")
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

## 1. Graph and literal semantics
Extension of the rdflib graph model to support RDF 1.2.  
- Triple terms and statement resources
- Reification via rdf:reifies and statement metadata
- Direction-tagged strings such as "hello"@en--ltr and "مرحبا"@ar--rtl

In [18]:
#create the graph, and assign namespace
g = StarLayerGraph()
g.bind("ex", EX)

#create a triple term
tt = TripleTerm(EX.bob, EX.knows, EX.carol)

#create a reifer associated with triple term and add to graph
g.add_reification(EX.claim, tt)
g.add((EX.claim, EX.source, EX.wikipedia))

print((EX.claim, RDF.reifies, tt) in g)
print(g.serialize(format="turtle12"))

True
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim ex:source ex:wikipedia ;
    rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



In [26]:
g = StarLayerGraph()
g.bind("ex", EX)

#add reifiers to the graph
g.add((EX.claim, RDF.reifies, (EX.bob, EX.knows, EX.carol)))
g.add((EX.other, RDF.reifies, (EX.bob, EX.likes, EX.dana)))

#triples accepts triple term as object to select triples
selectTriples = g.triples((None, None, (EX.bob, EX.knows, EX.carol)))

for s, p, o in selectTriples:
    print(g.qname(s), g.qname(p), o)
for t in g.triple_terms(subject=EX.bob):
    print(t)
print(g.has_triple_term(EX.bob, EX.knows, EX.carol))
print(g.has_triple_term(EX.bob, EX.knows, EX.dana))

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:likes ex:dana )>>
True
False


In [27]:
g = StarLayerGraph()
g.bind("ex", EX)

#literals can include language direction.
g.add((EX.title, EX.value, DirLangString("مرحبا", "ar", "rtl")))
print(g.serialize(format="turtle12"))

@version "1.2" .
@prefix ex: <http://example.org/> .

ex:title ex:value "مرحبا"@ar--rtl .



## 2. SPARQL and query semantics
Extension of graph semantics into SPARQL query execution and RDF 1.2 parsing behavior.
- RDF 1.2-aware query handling over triple terms and direction-tagged literals
- SPARQL support for isTRIPLE and triple-term patterns
- Turtle/SPARQL parsing and serialization support

In [ ]:
g = StarLayerGraph()

# parse RDF 1.2 Turtle data containing a direction-tagged literal
g.parse(data='''
    @prefix ex: <http://example.org/> .
    ex:note ex:text "مرحبا"@ar--rtl .
''', format='turtle12')

# serialize back to turtle12 to confirm round-trip behavior
print(g.serialize(format='turtle12'))

### Troubleshooting: missing sparql_shapes.ttl

If query cells fail with `FileNotFoundError` for `starsparql/ontology/sparql_shapes.ttl`, your installed wheel was built without package data.

Run this in the notebook and restart the kernel:

```python
!pip uninstall -y starlayer sparql graph shacl
!pip install --no-cache-dir "git+https://github.com/hidden-graph/starlayer.git"
```

In [ ]:
g = StarLayerGraph()
g.bind("ex", EX)

# add a reified triple-term statement and metadata
g.add((EX.claim, RDF.reifies, (EX.bob, EX.knows, EX.carol)))
g.add((EX.claim, EX.source, EX.wikipedia))

# query directly using triple-term syntax in the WHERE clause
rows = g.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim ?source WHERE {
  ?claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .
  ?claim ex:source ?source .
}
""")

for row in rows:
    print(g.qname(row.claim), g.qname(row.source))

In [ ]:
# filter for bindings where the object is a triple term
rows = g.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim WHERE {
  ?claim rdf:reifies ?statement .
  FILTER( isTRIPLE(?statement) )
}
""")

for row in rows:
    print(g.qname(row.claim))

## 3. SHACL validation and rules

- Validation over RDF 1.2 graphs
- SHACL rule support
- Direction-aware datatype constraints

In [ ]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ;
      ex:age 30 .
""", format="turtle")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:age ; sh:minCount 1 ; sh:datatype xsd:integer ] .
""", format="turtle")
result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes)
print(result.conforms)

In [ ]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
""", format="turtle")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonRule a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:rule [ a sh:TripleRule ; sh:subject sh:this ; sh:predicate ex:inferred ; sh:object ex:yes ] .
""", format="turtle")
result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print((EX.alice, EX.inferred, EX.yes) in result.data_graph)
print(result.conforms)

## 4. Backend graph-store and format support

- In-memory backend and native RDF 1.2 backend modes
- Eight RDF 1.2-aware formats: turtle12, nt12, nq12, trig12, trix12, rdfxml12, jsonld12, longturtle12
- Dual-mode RDF 1.1 and RDF 1.2 operation

In [ ]:
g = StarLayerGraph()
g.bind("ex", EX)
g.add((EX.alice, EX.claims, (EX.bob, EX.knows, EX.carol)))
print(g.serialize(format="turtle12"))
print("---")
print(g.serialize(format="nt12"))

### Switching to the native backend

```python
g = StarLayerGraph()
g = StarLayerGraph(backend='rdf-1.2')
```

## Summary

This notebook now tracks the same major points as docs/user-guide-v1.md.